# 02 - Metric Driver Slices

This notebook recreates the simplest plots behind `metric_analysis.qmd` and `metric_analysis.tex`.
Each slice varies one Class 1 assumption while Class 2 stays fixed at baseline.

Use this to understand how to create new one-parameter sensitivity checks without editing the production figure generator.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_DIR = Path.cwd()
if not (REPO_DIR / "engine_files").exists():
    REPO_DIR = REPO_DIR.parent
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import (
    METRIC_DEFINITIONS,
    aggregate_result_row,
    class_result_rows,
    metric_definition_rows,
    outcome_rates_from_result,
    outcome_totals,
    result_metrics_from_result,
)
from analysis.plot_style import (
    BASELINE_COLOR,
    driver_heatmap_cmap,
    driver_line_style,
    plot_driver_line,
)
from engine_files.config_loader import load_config
from engine_files.engine import ClinicAppointmentSimulation
from engine_files.model import ThresholdRule

BASE_CONFIG = load_config(REPO_DIR / "configs" / "baseline.yaml")
pd.options.display.max_columns = 120

## Config Mutation Helpers

These helpers show how to build new scenarios without mutating the loaded baseline config.

In [ ]:
def make_step_rule(old_rule: ThresholdRule, threshold=None, step=None) -> ThresholdRule:
    threshold = old_rule.threshold if threshold is None else int(threshold)
    low = old_rule.low
    step = old_rule.high - old_rule.low if step is None else float(step)
    return ThresholdRule(threshold=threshold, low=low, high=min(low + step, 1.0))


def update_classes(config, changes):
    classes = {}
    for class_id, params in config.classes.items():
        classes[class_id] = replace(params, **changes.get(class_id, {}))
    return replace(config, classes=classes)


def set_class1_balk_step(config, step):
    return update_classes(config, {1: {"balk_prob": make_step_rule(config.classes[1].balk_prob, step=step)}})


def set_class1_balk_threshold(config, threshold):
    return update_classes(config, {1: {"balk_prob": make_step_rule(config.classes[1].balk_prob, threshold=threshold)}})


def set_class1_no_show_step(config, step):
    return update_classes(config, {1: {"no_show_prob": make_step_rule(config.classes[1].no_show_prob, step=step)}})


def set_class1_cancel_prob(config, cancel_prob):
    return update_classes(config, {1: {"cancel_prob": float(cancel_prob)}})


def set_total_arrival_multiplier(config, multiplier):
    classes = {
        class_id: replace(params, lambda_per_day=params.lambda_per_day * float(multiplier))
        for class_id, params in config.classes.items()
    }
    return replace(config, classes=classes)

## Slice Runner

For exploration, start with a few seeds. Increase `SEEDS` when you want smoother curves.

In [ ]:
SEEDS = [2027, 2028, 2029]


def run_mean_metrics(config_builder, x_values, x_name, seeds=SEEDS):
    rows = []
    for x in x_values:
        for seed in seeds:
            config = replace(config_builder(BASE_CONFIG, x), seed=seed)
            result = ClinicAppointmentSimulation(config).run()
            rows.append({x_name: x, "seed": seed, **result_metrics_from_result(result)})
    return pd.DataFrame(rows).groupby(x_name, as_index=False).mean(numeric_only=True)


def plot_slice(df, x_name, panels, driver):
    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4), squeeze=False)
    for ax, panel in zip(axes[0], panels):
        for index, (metric, label) in enumerate(panel["series"]):
            plot_driver_line(ax, df[x_name], df[metric], label, driver=driver, index=index)
        ax.set_title(panel["title"])
        ax.set_xlabel(panel.get("xlabel", x_name))
        ax.set_ylabel(panel.get("ylabel", "value"))
        ax.grid(True, alpha=0.3)
        if "baseline" in panel:
            ax.axvline(panel["baseline"], color=BASELINE_COLOR, linestyle="--", linewidth=1.2, label="baseline")
        ax.legend(frameon=False)
    fig.tight_layout()
    return fig

## Balking Step Slice

This is the simplest way to understand the balking driver plots: Class 1 changes, Class 2 stays fixed.

In [ ]:
balk_step_values = np.linspace(0.0, 1.0, 11)
baseline_balk_step = BASE_CONFIG.classes[1].balk_prob.high - BASE_CONFIG.classes[1].balk_prob.low

balk_step_df = run_mean_metrics(set_class1_balk_step, balk_step_values, "class_1_balk_step")
display(balk_step_df)

plot_slice(
    balk_step_df,
    "class_1_balk_step",
    [
        {
            "title": "Served rate",
            "ylabel": "served / arrivals",
            "baseline": baseline_balk_step,
            "series": [
                ("overall_percent_serviced", "overall"),
                ("class_1_percent_serviced", "Class 1"),
                ("class_2_percent_serviced", "Class 2"),
            ],
        },
        {
            "title": "Balking rate",
            "ylabel": "balked / offered",
            "baseline": baseline_balk_step,
            "series": [
                ("overall_balking_rate", "overall"),
                ("class_1_balking_rate", "Class 1"),
                ("class_2_balking_rate", "Class 2"),
            ],
        },
    ],
    driver="balking",
)

## Compare Multiple Drivers

Change the `DRIVER_RUNS` dictionary to add new slices. This cell gives a compact comparison table for the report's main metrics.

In [ ]:
DRIVER_RUNS = {
    "balking_step": (set_class1_balk_step, np.linspace(0.0, 1.0, 6), "balking"),
    "balking_threshold": (set_class1_balk_threshold, list(range(0, BASE_CONFIG.horizon_days, 2)), "balking"),
    "no_show_step": (set_class1_no_show_step, np.linspace(0.0, 1.0, 6), "no_show"),
    "cancel_prob": (set_class1_cancel_prob, np.linspace(0.0, 0.30, 6), "cancellation"),
    "arrival_multiplier": (set_total_arrival_multiplier, np.linspace(0.5, 1.5, 6), "arrival"),
}

summary_rows = []
for name, (builder, values, driver) in DRIVER_RUNS.items():
    df = run_mean_metrics(builder, values, name)
    first = df.iloc[0]
    last = df.iloc[-1]
    summary_rows.append(
        {
            "driver": name,
            "x_start": first[name],
            "x_end": last[name],
            "delta_utilization": last.average_utilization - first.average_utilization,
            "delta_access": last.overall_percent_serviced - first.overall_percent_serviced,
            "delta_offered_wait": last.mean_offered_booking_delay - first.mean_offered_booking_delay,
            "delta_class_gap": last.access_advantage_class_1 - first.access_advantage_class_1,
        }
    )

pd.DataFrame(summary_rows).sort_values("delta_access")

## Optional Widget Explorer

If `ipywidgets` is available in your environment, this cell creates a small interactive single-scenario explorer. If not, the plain function call still works.

In [ ]:
def run_one_point(class1_balk_step=0.50, class1_threshold=9, class1_cancel=0.10, arrival_multiplier=1.0, seed=2027):
    config = set_class1_balk_step(BASE_CONFIG, class1_balk_step)
    config = set_class1_balk_threshold(config, class1_threshold)
    config = set_class1_cancel_prob(config, class1_cancel)
    config = set_total_arrival_multiplier(config, arrival_multiplier)
    config = replace(config, seed=int(seed))
    result = ClinicAppointmentSimulation(config).run()
    return pd.Series(result_metrics_from_result(result)).to_frame("value")

try:
    import ipywidgets as widgets
    from IPython.display import display

    explorer = widgets.interactive(
        run_one_point,
        class1_balk_step=(0.0, 1.0, 0.05),
        class1_threshold=(0, BASE_CONFIG.horizon_days - 1, 1),
        class1_cancel=(0.0, 0.30, 0.01),
        arrival_multiplier=(0.5, 1.5, 0.05),
        seed=(1, 5000, 1),
    )
    display(explorer)
except ImportError:
    display(run_one_point())